# SciCite Extension: ToW-MLM Adaptation

This notebook runs the **ToW-MLM Adaptation** experiment for the ToW reproducibility report.

**Goal.** Test whether clean word-level Thoughts of Words (ToW) can improve SciCite citation-intent classification when used as a training-time representation-learning signal rather than as extra text at test time.

**Procedure.**

1. Load cached raw SciCite sample splits.
2. Load cached clean word-level ToW splits.
3. Continue masked-language-model (MLM) adaptation of SciBERT on clean ToW-augmented citation contexts.
4. Fine-tune and evaluate two classifiers on **raw citation text only**:
   - Raw SciBERT baseline.
   - ToW-MLM-adapted SciBERT.
5. Save metrics, predictions, the adapted checkpoint, and a paper-ready figure.

This notebook does **not** generate ToW annotations and does **not** call the OpenAI API.


In [ ]:

# ============================================================
# 0. Optional dependency setup
# ============================================================
# Run this once in a fresh Colab runtime. It avoids vLLM/DeepSpeed because this notebook
# only needs Hugging Face training for SciBERT.

import sys, subprocess, importlib.util

def has_pkg(name):
    return importlib.util.find_spec(name) is not None

needed = ["transformers", "datasets", "evaluate", "sklearn", "torch"]
missing = [p for p in needed if not has_pkg("scikit-learn" if p == "sklearn" else p)]

if missing:
    print("Installing missing packages:", missing)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers>=4.43.0,<5.0.0",
        "datasets>=2.19.0,<4.0.0",
        "evaluate>=0.4.0",
        "scikit-learn>=1.3.0",
        "accelerate>=0.30.0",
        "matplotlib",
        "pandas",
    ], check=True)
else:
    print("Core packages already available.")

In [ ]:
# ============================================================
# 1. Configuration, Drive mount, and file discovery
# ============================================================

from pathlib import Path
import os, json, random, inspect, math, shutil, time
import numpy as np
import pandas as pd
import torch

# Mount Google Drive when running in Colab.
try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    else:
        print("Drive already mounted.")
except Exception as e:
    print("Drive mount skipped/warning:", repr(e))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Expected/default paths. If these do not exist, the notebook auto-discovers them below.
DEFAULT_BASE_DIR = Path("/content/drive/MyDrive/tow_project/B/extension/scicite")
DEFAULT_DATA_DIR = DEFAULT_BASE_DIR / "data"
DEFAULT_WORDLEVEL_DIR = DEFAULT_BASE_DIR / "wordlevel_tow_generations_clean_v2"

SPLITS = ["train", "validation", "test"]

def _count_jsonl_quick(path):
    path = Path(path)
    if not path.exists():
        return 0
    n = 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                n += 1
    return n

def _complete_dir(parent, pattern_template):
    parent = Path(parent)
    return all((parent / pattern_template.format(split=s)).exists() for s in SPLITS)

def _find_complete_parent(search_root, train_filename, pattern_template, prefer_keywords=()):
    search_root = Path(search_root)
    if not search_root.exists():
        return None
    hits = sorted(search_root.rglob(train_filename))
    parents = sorted({p.parent for p in hits})
    complete = [p for p in parents if _complete_dir(p, pattern_template)]
    if not complete:
        return None

    def score(p):
        s = str(p).lower()
        # Prefer exact/clean/newer project paths.
        val = 0
        for i, kw in enumerate(prefer_keywords):
            if kw.lower() in s:
                val += 100 - i
        # Prefer folders with more total rows.
        val += sum(_count_jsonl_quick(p / pattern_template.format(split=sp)) for sp in SPLITS) / 10000
        return val

    complete = sorted(complete, key=score, reverse=True)
    return complete[0]

# Find raw data directory.
if _complete_dir(DEFAULT_DATA_DIR, "scicite_{split}_raw_sample.jsonl"):
    DATA_DIR = DEFAULT_DATA_DIR
else:
    DATA_DIR = _find_complete_parent(
        "/content/drive/MyDrive",
        "scicite_train_raw_sample.jsonl",
        "scicite_{split}_raw_sample.jsonl",
        prefer_keywords=("tow_project/b/extension/scicite/data", "tow_project", "scicite"),
    )
    if DATA_DIR is None:
        DATA_DIR = _find_complete_parent(
            "/content/drive/MyDrive",
            "scicite_train_raw.jsonl",
            "scicite_{split}_raw.jsonl",
            prefer_keywords=("tow_project/b/extension/scicite/data", "tow_project", "scicite"),
        )

# Find clean word-level ToW directory.
if _complete_dir(DEFAULT_WORDLEVEL_DIR, "scicite_{split}_wordlevel_tow.jsonl"):
    WORDLEVEL_DIR = DEFAULT_WORDLEVEL_DIR
else:
    WORDLEVEL_DIR = _find_complete_parent(
        "/content/drive/MyDrive",
        "scicite_train_wordlevel_tow.jsonl",
        "scicite_{split}_wordlevel_tow.jsonl",
        prefer_keywords=("clean_v2", "wordlevel_tow_generations_clean_v2", "clean", "tow_project/b/extension/scicite", "wordlevel"),
    )

if DATA_DIR is None:
    raise RuntimeError("Could not find raw SciCite jsonl files in Drive. Mount Drive and confirm scicite_train_raw_sample.jsonl exists.")
if WORDLEVEL_DIR is None:
    raise RuntimeError("Could not find clean word-level ToW jsonl files in Drive. Confirm the clean word-level ToW generator finished and saved all splits.")

# Pick a project root for results. Usually DATA_DIR.parent is .../B/extension/scicite.
BASE_DIR = DATA_DIR.parent if DATA_DIR.name == "data" else DEFAULT_BASE_DIR
if not BASE_DIR.exists():
    BASE_DIR = WORDLEVEL_DIR.parent

# ToW-MLM output paths. Uses a separate folder so this does not overwrite direct-input extension results.
RESULTS_DIR = BASE_DIR / "results_tow_mlm_adaptation_clean_v2"
MLM_MODEL_DIR = RESULTS_DIR / "mlm_adapted_scibert_clean_wordlevel_tow"
FIG_DIR = RESULTS_DIR / "figures"
PRED_DIR = RESULTS_DIR / "predictions"
for p in [RESULTS_DIR, FIG_DIR, PRED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "allenai/scibert_scivocab_uncased"
NUM_LABELS = 3
LABEL_NAMES = ["background", "method", "result"]

# MLM adaptation settings. This is still quick because the dataset is tiny.
MLM_EPOCHS = 10
MLM_LR = 5e-5
MLM_BATCH_SIZE = 8
MLM_PROBABILITY = 0.15

# Classification settings.
CLS_EPOCHS = 10
CLS_LR = 2e-5
CLS_BATCH_SIZE = 8
WEIGHT_DECAY = 0.01
MAX_LENGTH = 384

# Fair comparison: restrict both conditions to IDs present in raw and clean word-level ToW files.
USE_COMMON_IDS = True

# Set these to False only if you already ran the exact same output folder and want to reuse it.
RUN_MLM_ADAPTATION = True
RUN_CLASSIFIER_TRAINING = True

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", DATA_DIR, "exists=", DATA_DIR.exists())
print("WORDLEVEL_DIR:", WORDLEVEL_DIR, "exists=", WORDLEVEL_DIR.exists())
print("RESULTS_DIR:", RESULTS_DIR)


In [ ]:
# ============================================================
# 2. Basic JSONL helpers and file checks
# ============================================================

def read_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def write_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def write_jsonl(rows, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def raw_file_for(split):
    candidates = [
        DATA_DIR / f"scicite_{split}_raw_sample.jsonl",
        DATA_DIR / f"scicite_{split}_raw.jsonl",
        BASE_DIR / f"scicite_{split}_raw_sample.jsonl",
    ]
    for p in candidates:
        if Path(p).exists():
            return Path(p)
    return candidates[0]

def wordlevel_file_for(split):
    candidates = [
        WORDLEVEL_DIR / f"scicite_{split}_wordlevel_tow.jsonl",
        BASE_DIR / "wordlevel_tow_generations_clean_v2" / f"scicite_{split}_wordlevel_tow.jsonl",
        BASE_DIR / "wordlevel_tow_generations_clean" / f"scicite_{split}_wordlevel_tow.jsonl",
        BASE_DIR / "wordlevel_tow_generations" / f"scicite_{split}_wordlevel_tow.jsonl",
    ]
    for p in candidates:
        if Path(p).exists():
            return Path(p)
    return candidates[0]

print("Raw data files:")
for split in ["train", "validation", "test"]:
    p = raw_file_for(split)
    print(split, p.exists(), len(read_jsonl(p)), p)

print("\nClean word-level ToW files:")
for split in ["train", "validation", "test"]:
    p = wordlevel_file_for(split)
    print(split, p.exists(), len(read_jsonl(p)), p)

missing = []
for split in ["train", "validation", "test"]:
    if not raw_file_for(split).exists():
        missing.append(str(raw_file_for(split)))
    if not wordlevel_file_for(split).exists():
        missing.append(str(wordlevel_file_for(split)))
if missing:
    raise RuntimeError("Missing required files after auto-discovery:\n" + "\n".join(missing))


In [ ]:

# ============================================================
# 3. Load raw SciCite and clean word-level ToW rows
# ============================================================

from datasets import Dataset, DatasetDict

VALID_WORDLEVEL_CATEGORIES = {"exact_match", "soft_consistent"}

def normalize_label(x):
    # Handles int labels and common SciCite string label variants.
    if isinstance(x, int):
        return x
    if isinstance(x, str):
        lx = x.lower().strip()
        mapping = {
            "background": 0,
            "method": 1,
            "result": 2,
            "result_comparison": 2,
            "result-comparison": 2,
        }
        if lx in mapping:
            return mapping[lx]
        if lx.isdigit():
            return int(lx)
    raise ValueError(f"Cannot normalize label: {x!r}")

def valid_raw_row(r):
    return (
        isinstance(r, dict)
        and isinstance(r.get("id"), str)
        and isinstance(r.get("text"), str)
        and len(r.get("text", "").strip()) > 0
        and r.get("label") is not None
    )

def valid_clean_wordlevel_row(r):
    aug = r.get("tow_augmented_text", "")
    cat = r.get("tow_category", "")
    return (
        isinstance(r, dict)
        and isinstance(r.get("id"), str)
        and isinstance(r.get("text"), str)
        and r.get("label") is not None
        and isinstance(aug, str)
        and "<ToW>" in aug
        and "</ToW>" in aug
        and (not cat or cat in VALID_WORDLEVEL_CATEGORIES)
    )

raw_by_split = {}
word_by_split = {}
common_ids = {}

for split in ["train", "validation", "test"]:
    raw_rows = [r for r in read_jsonl(raw_file_for(split)) if valid_raw_row(r)]
    word_rows = [r for r in read_jsonl(wordlevel_file_for(split)) if valid_clean_wordlevel_row(r)]

    raw_by_split[split] = {r["id"]: r for r in raw_rows}
    word_by_split[split] = {r["id"]: r for r in word_rows}
    common_ids[split] = sorted(set(raw_by_split[split]) & set(word_by_split[split]))

    print(split)
    print("  valid raw:", len(raw_by_split[split]))
    print("  valid clean word-level:", len(word_by_split[split]))
    print("  common ids:", len(common_ids[split]))

if USE_COMMON_IDS:
    for split in ["train", "validation", "test"]:
        if len(common_ids[split]) == 0:
            raise RuntimeError(f"No common IDs for split={split}.")

# Save the exact IDs used
write_json(common_ids, RESULTS_DIR / "common_ids_used.json")

In [ ]:

# ============================================================
# 4. Build the MLM adaptation dataset and raw-classification datasets
# ============================================================

def rows_for_split(split, source="raw"):
    ids = common_ids[split] if USE_COMMON_IDS else sorted(raw_by_split[split])
    rows = []
    for rid in ids:
        raw = raw_by_split[split].get(rid)
        word = word_by_split[split].get(rid)
        if raw is None:
            continue
        if source == "raw":
            rows.append({
                "id": rid,
                "text": "Citation: " + raw["text"],
                "label": normalize_label(raw["label"]),
                "label_name": raw.get("label_name"),
            })
        elif source == "wordlevel_mlm":
            if word is None:
                continue
            rows.append({
                "id": rid,
                "text": "Citation: " + word["tow_augmented_text"],
            })
        else:
            raise ValueError(source)
    return rows

raw_cls_dd = DatasetDict({
    split: Dataset.from_list(rows_for_split(split, "raw"))
    for split in ["train", "validation", "test"]
})

mlm_dd = DatasetDict({
    "train": Dataset.from_list(rows_for_split("train", "wordlevel_mlm")),
    "validation": Dataset.from_list(rows_for_split("validation", "wordlevel_mlm")),
})

print("Raw classifier dataset:")
print(raw_cls_dd)
print("\nClean word-level ToW MLM dataset:")
print(mlm_dd)
print("\nMLM example:")
print(mlm_dd["train"][0]["text"][:1200])

# Export a few examples
write_jsonl([mlm_dd["train"][i] for i in range(min(10, len(mlm_dd["train"])))], RESULTS_DIR / "mlm_train_examples.jsonl")


In [ ]:

# ============================================================
# 5. Training helper functions
# ============================================================

from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    AutoModelForSequenceClassification,
    DataCollatorForLanguageModeling,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

set_seed(SEED)
SPECIAL_TOKENS = ["<ToW>", "</ToW>"]

def filter_training_args_kwargs(kwargs):
    sig = inspect.signature(TrainingArguments.__init__)
    return {k: v for k, v in kwargs.items() if k in sig.parameters}

def strategy_key():
    sig = inspect.signature(TrainingArguments.__init__)
    return "eval_strategy" if "eval_strategy" in sig.parameters else "evaluation_strategy"

def make_args(
    output_dir,
    epochs,
    lr,
    batch_size,
    weight_decay=0.0,
    metric_for_best_model=None,
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=10,
):
    kwargs = dict(
        output_dir=str(output_dir),
        num_train_epochs=epochs,
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        weight_decay=weight_decay,
        logging_strategy="epoch",
        save_strategy="epoch",
        seed=SEED,
        data_seed=SEED,
        report_to=[],
        save_total_limit=save_total_limit,
        load_best_model_at_end=metric_for_best_model is not None,
        metric_for_best_model=metric_for_best_model,
        greater_is_better=greater_is_better,
        fp16=torch.cuda.is_available(),
    )
    kwargs[strategy_key()] = "epoch"
    return TrainingArguments(**filter_training_args_kwargs(kwargs))

def safe_trainer(**kwargs):
    # Newer Transformers may reject tokenizer=..., older versions accept it.
    try:
        return Trainer(**kwargs)
    except TypeError as e:
        if "tokenizer" in str(e):
            kwargs.pop("tokenizer", None)
            return Trainer(**kwargs)
        raise

def compute_cls_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    metrics = {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
    }
    per_class = f1_score(labels, preds, average=None, labels=[0, 1, 2], zero_division=0)
    for name, val in zip(LABEL_NAMES, per_class):
        metrics[f"f1_{name}"] = float(val)
    return metrics

print("Training helpers loaded.")

In [ ]:

# ============================================================
# 6. Continue MLM adaptation of SciBERT on clean word-level ToW text
# ============================================================

if RUN_MLM_ADAPTATION:
    print("Running ToW-MLM adaptation...")
    tokenizer_mlm = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenizer_mlm.add_special_tokens({"additional_special_tokens": SPECIAL_TOKENS})

    def tokenize_mlm(batch):
        return tokenizer_mlm(
            batch["text"],
            truncation=True,
            max_length=MAX_LENGTH,
        )

    tokenized_mlm = mlm_dd.map(tokenize_mlm, batched=True, remove_columns=mlm_dd["train"].column_names)
    data_collator_mlm = DataCollatorForLanguageModeling(
        tokenizer=tokenizer_mlm,
        mlm=True,
        mlm_probability=MLM_PROBABILITY,
    )

    mlm_model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)
    mlm_model.resize_token_embeddings(len(tokenizer_mlm))

    mlm_args = make_args(
        output_dir=RESULTS_DIR / "mlm_training_runs",
        epochs=MLM_EPOCHS,
        lr=MLM_LR,
        batch_size=MLM_BATCH_SIZE,
        weight_decay=0.01,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
    )

    mlm_trainer = safe_trainer(
        model=mlm_model,
        args=mlm_args,
        train_dataset=tokenized_mlm["train"],
        eval_dataset=tokenized_mlm["validation"],
        data_collator=data_collator_mlm,
        tokenizer=tokenizer_mlm,
    )

    mlm_train_result = mlm_trainer.train()
    mlm_eval = mlm_trainer.evaluate()
    print("MLM eval:", mlm_eval)

    MLM_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    mlm_trainer.save_model(str(MLM_MODEL_DIR))
    tokenizer_mlm.save_pretrained(str(MLM_MODEL_DIR))
    write_json({"train_result": str(mlm_train_result), "eval": mlm_eval}, RESULTS_DIR / "mlm_adaptation_metrics.json")
    print("Saved ToW-MLM-adapted model to:", MLM_MODEL_DIR)
else:
    print("Skipping MLM adaptation. Expecting existing model at:", MLM_MODEL_DIR)
    if not MLM_MODEL_DIR.exists():
        raise RuntimeError("MLM_MODEL_DIR does not exist. Set RUN_MLM_ADAPTATION=True or check path.")

In [ ]:

# ============================================================
# 7. Fine-tune/evaluate raw baseline vs. ToW-MLM-adapted SciBERT
#    Both classifiers train and test on raw SciCite text only.
# ============================================================

from copy import deepcopy

CONDITIONS = [
    {
        "condition": "baseline_raw_scibert",
        "description": "Raw SciBERT fine-tuned/evaluated on raw citation context",
        "model_path": MODEL_NAME,
        "tokenizer_path": MODEL_NAME,
    },
    {
        "condition": "tow_mlm_adaptation",
        "description": "ToW-MLM Adaptation: SciBERT adapted with MLM on clean word-level ToW, then fine-tuned/evaluated on raw citation context",
        "model_path": str(MLM_MODEL_DIR),
        "tokenizer_path": str(MLM_MODEL_DIR),
    },
]

def tokenize_cls_dataset(tokenizer, dd):
    def tok(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=MAX_LENGTH,
        )
    return dd.map(tok, batched=True)

def train_and_evaluate_cls(condition, description, model_path, tokenizer_path):
    print(f"\n=== Training/evaluating {condition} ===")
    output_dir = RESULTS_DIR / "classifier_runs" / condition
    metrics_path = RESULTS_DIR / f"{condition}_metrics.json"
    preds_path = PRED_DIR / f"{condition}_test_predictions.jsonl"

    if (not RUN_CLASSIFIER_TRAINING) and metrics_path.exists():
        print("Skipping; metrics already exist:", metrics_path)
        with open(metrics_path, "r", encoding="utf-8") as f:
            return json.load(f)

    tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_path,
        num_labels=NUM_LABELS,
        ignore_mismatched_sizes=True,
    )

    encoded = tokenize_cls_dataset(tokenizer, raw_cls_dd)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    cls_args = make_args(
        output_dir=output_dir,
        epochs=CLS_EPOCHS,
        lr=CLS_LR,
        batch_size=CLS_BATCH_SIZE,
        weight_decay=WEIGHT_DECAY,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
    )

    trainer = safe_trainer(
        model=model,
        args=cls_args,
        train_dataset=encoded["train"],
        eval_dataset=encoded["validation"],
        data_collator=data_collator,
        tokenizer=tokenizer,
        compute_metrics=compute_cls_metrics,
    )

    trainer.train()
    test_output = trainer.predict(encoded["test"])
    logits = test_output.predictions
    labels = test_output.label_ids
    preds = np.argmax(logits, axis=-1)

    metrics = compute_cls_metrics((logits, labels))
    row = {
        "condition": condition,
        "description": description,
        "test_accuracy": float(metrics["accuracy"]),
        "test_macro_f1": float(metrics["macro_f1"]),
        "num_train": len(raw_cls_dd["train"]),
        "num_validation": len(raw_cls_dd["validation"]),
        "num_test": len(raw_cls_dd["test"]),
        "classifier_model_or_init": model_path,
        "base_model": MODEL_NAME,
        "mlm_epochs_for_tow_mlm": MLM_EPOCHS if condition == "tow_mlm_adaptation" else 0,
        "classifier_epochs": CLS_EPOCHS,
        "max_length": MAX_LENGTH,
        "use_common_ids": USE_COMMON_IDS,
        "f1_background": float(metrics["f1_background"]),
        "f1_method": float(metrics["f1_method"]),
        "f1_result": float(metrics["f1_result"]),
    }

    write_json(row, metrics_path)

    pred_rows = []
    test_rows = raw_cls_dd["test"]
    for i in range(len(test_rows)):
        pred_rows.append({
            "id": test_rows[i]["id"],
            "text": test_rows[i]["text"],
            "gold_label": int(labels[i]),
            "gold_label_name": LABEL_NAMES[int(labels[i])],
            "pred_label": int(preds[i]),
            "pred_label_name": LABEL_NAMES[int(preds[i])],
        })
    write_jsonl(pred_rows, preds_path)

    print("Saved metrics:", metrics_path)
    print("Saved predictions:", preds_path)
    print(row)
    return row

results = []
for c in CONDITIONS:
    results.append(train_and_evaluate_cls(**c))

print("\nDone.")


In [ ]:

# ============================================================
# 8. Save ToW-MLM results table and figure
# ============================================================

import matplotlib.pyplot as plt

results_df = pd.DataFrame(results)
results_df = results_df.sort_values("condition")

csv_path = RESULTS_DIR / "scicite_extension_tow_mlm_results.csv"
json_path = RESULTS_DIR / "scicite_extension_tow_mlm_results.json"
results_df.to_csv(csv_path, index=False)
write_json(results, json_path)

print("Saved:", csv_path)
print("Saved:", json_path)
display(results_df)

# Compact paper figure.
fig_df = results_df.copy()
fig_df["display"] = fig_df["condition"].map({
    "baseline_raw_scibert": "Raw SciBERT",
    "tow_mlm_adaptation": "ToW-MLM → Raw CLS",
}).fillna(fig_df["condition"])

x = np.arange(len(fig_df))
width = 0.35
plt.figure(figsize=(7, 4))
plt.bar(x - width/2, fig_df["test_accuracy"], width, label="Accuracy")
plt.bar(x + width/2, fig_df["test_macro_f1"], width, label="Macro F1")
plt.xticks(x, fig_df["display"], rotation=15, ha="right")
plt.ylim(0, 1.0)
plt.ylabel("Score")
plt.title("SciCite ToW-MLM Adaptation")
plt.legend()
plt.tight_layout()
fig_path = FIG_DIR / "scicite_extension_tow_mlm_scores.pdf"
plt.savefig(fig_path)
plt.show()
print("Saved figure:", fig_path)

In [ ]:
# ============================================================
# 9. Optional: compare with direct-input ToW extension results if present
# ============================================================

candidate_direct_input_paths = [
    BASE_DIR / "results_direct_input_tow_extensions" / "scicite_direct_input_tow_results.csv",
]

direct_input_path = next((p for p in candidate_direct_input_paths if p.exists()), None)

if direct_input_path is not None:
    direct_input = pd.read_csv(direct_input_path)
    print("Found direct-input ToW extension results:", direct_input_path)
    display(direct_input)
    print("\nToW-MLM Adaptation results:")
    display(results_df)
else:
    print("No direct-input ToW extension results found.")
